In [5]:
import pennylane as qml
from pennylane import numpy as np
from scipy.optimize import minimize
from rl_qaoa import *
import random
# Define QAOA depth
depth =1
size = 12
seed = 50
hamming_weight = 6
penalty = 9

# Generate a QUBO matrix that is challenging for classical QAOA optimization
np.random.seed(seed)
Q = generate_upper_triangular_qubo(size, (-1,-3), (2,4), integer=False)
Q_cal = zero_lower_triangle(Q + qubo_to_ising(add_constraint([1]*size,hamming_weight)*penalty))
print(Q)

[[-2.59053903  2.45616621  2.51094785  2.79265982  2.7546302   3.99314846
   2.8163944   3.54378799  3.52107338  2.6200187   2.69308241  2.70352963]
 [ 0.         -1.07406683  3.81835688  3.11991421  2.62718149  3.77640008
   3.34914613  2.7821749   3.01436825  3.04820701  3.85600186  3.14274614]
 [ 0.          0.         -2.4690822   2.11280328  2.35965539  3.85186635
   3.87603045  3.42818541  3.46537522  2.92349535  3.86265854  2.81284049]
 [ 0.          0.          0.         -2.54514173  3.36471434  3.75611261
   3.59343453  2.8640045   3.83575644  3.56366736  3.45150056  2.24970939]
 [ 0.          0.          0.          0.         -1.48884905  2.51067783
   3.67799178  2.3572384   2.45422834  3.31975289  2.95823249  2.14689468]
 [ 0.          0.          0.          0.          0.         -2.97015372
   3.05332818  3.84452538  3.83851403  2.50401773  3.36526522  3.92855224]
 [ 0.          0.          0.          0.          0.          0.
  -2.35001169  2.06973066  2.02781073  3

In [13]:
class RL_QAOA_test(RL_QAOA):
    def RL_QAOA(self, episodes, epochs,log_interval =5, correct_ans=None):
        self.avg_values = []
        self.min_values = []
        self.prob_values = []
        self.best_states = []
        self.best_same_lists = []
        self.best_diff_lists = []
        
        """
        Performs the reinforcement learning optimization process with progress tracking.

        Parameters
        ----------
        episodes : int
            Number of Monte Carlo trials for the optimization.

        epochs : int
            Number of optimization iterations to update parameters.

        correct_ans : float, optional
            The correct optimal solution (if available) to calculate success probability.
        """
        break_value = False
        for j in range(epochs):
            if break_value:
                break
            
            if self.lr[0] != 0:
                num = self.tree.node_num
                
                self.tree = Tree('root',None)
                self.tree.node_num = num
                self.tree_grad = Tree('root',None)
                self.tree_grad.node_num = num
            value_list = []
            state_list = []
            QAOA_diff_list = []
            beta_diff_list = []
            same_lists = []
            diff_lists = []
            pass_num = 0
            if correct_ans is not None:
                prob = 0

            # Progress bar for episodes within the current epoch
            for i in tqdm(range(episodes), desc=f'Epoch {j + 1}/{epochs}', unit=' episode'):
                res = self.rqaoa_execute()
                QAOA_diff, beta_diff, value, final_state, same_list, diff_list = res
                value_list.append(value)
                state_list.append(final_state)
                same_lists.append(same_list)
                diff_lists.append(diff_list)
                QAOA_diff_list.append(QAOA_diff)
                beta_diff_list.append(beta_diff)

                if correct_ans is not None and correct_ans - 0.0001 <= value <= correct_ans + 0.0001:
                    prob += 1  




            # Compute softmax rewards and normalize

            batch_mean = (np.array(value_list) - np.mean(value_list))
            #batch_plus = np.where(batch_mean < 0, batch_mean, 0)
            #softmaxed_rewards = signed_softmax_rewards(batch_plus, beta=15)*episodes
            for index, val in enumerate(batch_mean):
                QAOA_diff_list[index] *= -batch_mean[index]*10
                beta_diff_list[index] *= -batch_mean[index]*10
                #QAOA_diff_list[index] *= value_list[index]
                #QAOA_diff_list[index] *= value_list[index]
            # Compute parameter updates
            QAOA_diff_sum = np.mean(QAOA_diff_list, axis=0)
            beta_diff_sum = np.mean(beta_diff_list, axis=0)
            value_sum = np.mean(value_list)
            min_value = np.min(value_list)  # Find the lowest reward value
            min_index = np.argmin(value_list)  # Index of lowest reward value
            # Store values
            self.avg_values.append(value_sum)
            self.min_values.append(min_value)
            if correct_ans is not None:
                prob /= episodes
            if prob > 0.5:
                break_value = True
                break

            self.prob_values.append(prob)
            self.best_states.append(state_list[min_index])
            self.best_same_lists.append(same_lists[min_index][:3])  # Store top 3 same list elements
            self.best_diff_lists.append(diff_lists[min_index][:3])  # Store top 3 diff list elements

            # Print optimization progress
            if j % log_interval == 0:
                if correct_ans is not None:
                    print(f'  Probability of finding correct solution: {prob:.4f}')
                print(f'  Average reward: {value_sum}')
                print(f'  Lowest reward obtained: {min_value}')
                print(f'  Best state at lowest value: {self.best_states[-1]}')
                print(f'  number of nodes : {self.tree.node_num}')

                #print(f'  Top 3 same constraints: {self.best_same_lists[-1]}')
                #print(f'  Top 3 different constraints: {self.best_diff_lists[-1]}')


            # Update parameters using the Adam optimizer
            self.optimzer.learning_rate = [self.lr[0],self.lr[1]]
            update = self.optimzer.get_updates([QAOA_diff_sum, beta_diff_sum])
            self.param += np.array(update[0])
            self.b += np.array(update[1])
            #self.optimzer.learning_rate = [self.lr[0]/20*j,self.lr[1]/20*j]

In [29]:
node_num_list = []
for rl_q in RL_qaoa_list:
    node_num_list.append(rl_q.tree.node_num)

In [30]:
node_num_list

[82,
 698,
 3092,
 8912,
 11729,
 19333,
 23662,
 26718,
 44261,
 172,
 1035,
 3730,
 12913,
 16062]

In [25]:
config_list = []
RL_qaoa_list = []
for size in range(6,15):
    # Define QAOA depth
    depth =1
    seed = 50
    hamming_weight = int(size/2)
    penalty = 9
    # Generate a QUBO matrix that is challenging for classical QAOA optimization
    np.random.seed(seed)
    Q = generate_upper_triangular_qubo(size, (-1,-3), (2,4), integer=False)
    Q_cal = zero_lower_triangle(Q + qubo_to_ising(add_constraint([1]*size,hamming_weight)*penalty))
    n = Q.shape[0]
    n_c = 3
    init_params = np.reshape(np.array([-0.12107375,  0.24919166]*(n-n_c)),-1)
    # RL-QAOA setup
    rl_qaoa = RL_QAOA_test(Q_cal,n,init_params,b_vector = np.array([[25.*((n-i)/10+1)]*int((n**2)) for i in range(n-n_c)]),QAOA_depth=depth,gamma = 0.99,learning_rate_init=[0.00,0.5])
    final_config = rl_qaoa.rqaoa_execute()
    rl_qaoa.n_c =n_c
    print(f"classical_result : {float(final_config[2])},best : {rl_qaoa.node_assignments}" )

    final_config = rl_qaoa.RL_QAOA(episodes=50,epochs=200,log_interval=10,correct_ans=float(final_config[2]))
    config_list.append(final_config)
    RL_qaoa_list.append(copy.deepcopy(rl_qaoa))
    print('\n\n\n')
    print(f'number of node : {rl_qaoa.tree.node_num}')
    print('\n\n\n')

classical_result : -20.879891861703026,best : [1, -1, 1, -1, 1, -1]


Epoch 1/200: 100%|██████████| 50/50 [00:01<00:00, 33.81 episode/s]






number of node : 82




classical_result : -28.087962609073962,best : [-1, 1, -1, 1, 1, 1, -1]


Epoch 1/200: 100%|██████████| 50/50 [00:03<00:00, 14.07 episode/s]


  Probability of finding correct solution: 0.2800
  Average reward: -25.699875475818477
  Lowest reward obtained: -28.087962609073962
  Best state at lowest value: [-1  1 -1  1  1  1 -1]
  number of nodes : 141


...

  Probability of finding correct solution: 0.0200
  Average reward: -49.4293191144064
  Lowest reward obtained: -56.17104973361969
  Best state at lowest value: [-1 -1 -1  1  1  1 -1  1 -1  1  1 -1 -1  1]
  number of nodes : 42417


Epoch 131/200: 100%|██████████| 50/50 [00:12<00:00,  4.15 episode/s]


  Probability of finding correct solution: 0.1600
  Average reward: -50.959877688693595
  Lowest reward obtained: -56.17104973361969
  Best state at lowest value: [-1 -1 -1  1  1  1 -1  1 -1  1  1 -1 -1  1]
  number of nodes : 43628


Epoch 138/200: 100%|██████████| 50/50 [00:08<00:00,  5.76 episode/s]






number of node : 44261






In [26]:
config_list_grad = []
RL_qaoa_list_grad = []
for size in range(6,15):
    # Define QAOA depth
    depth =1
    seed = 50
    hamming_weight = int(size/2)
    penalty = 9
    # Generate a QUBO matrix that is challenging for classical QAOA optimization
    np.random.seed(seed)
    Q = generate_upper_triangular_qubo(size, (-1,-3), (2,4), integer=False)
    Q_cal = zero_lower_triangle(Q + qubo_to_ising(add_constraint([1]*size,hamming_weight)*penalty))
    n = Q.shape[0]
    n_c = 3
    init_params = np.reshape(np.array([-0.12107375,  0.24919166]*(n-n_c)),-1)
    # RL-QAOA setup
    rl_qaoa = RL_QAOA_test(Q_cal,n,init_params,b_vector = np.array([[25.*((n-i)/10+1)]*int((n**2)) for i in range(n-n_c)]),QAOA_depth=depth,gamma = 0.99,learning_rate_init=[0.002,0.5])
    final_config = rl_qaoa.rqaoa_execute()
    rl_qaoa.n_c =n_c
    print(f"classical_result : {float(final_config[2])},best : {rl_qaoa.node_assignments}" )

    final_config = rl_qaoa.RL_QAOA(episodes=50,epochs=200,log_interval=10,correct_ans=float(final_config[2]))
    config_list.append(final_config)
    RL_qaoa_list.append(copy.deepcopy(rl_qaoa))
    print('\n\n\n')
    print(f'number of node : {rl_qaoa.tree.node_num}')
    print('\n\n\n')

classical_result : -20.879891861703026,best : [1, -1, 1, -1, 1, -1]


Epoch 1/200: 100%|██████████| 50/50 [00:05<00:00,  8.34 episode/s]


  Probability of finding correct solution: 0.4800
  Average reward: -19.04477502146028
  Lowest reward obtained: -20.879891861703026
  Best state at lowest value: [ 1 -1  1 -1  1 -1]
  number of nodes : 86


Epoch 2/200: 100%|██████████| 50/50 [00:05<00:00,  8.73 episode/s]






number of node : 172




classical_result : -28.087962609073962,best : [-1, 1, -1, 1, 1, 1, -1]


...

  Probability of finding correct solution: 0.1400
  Average reward: -40.541196498071685
  Lowest reward obtained: -47.019716095227274
  Best state at lowest value: [-1 -1  1  1 -1  1  1  1 -1 -1  1]
  number of nodes : 21917


Epoch 71/200: 100%|██████████| 50/50 [03:41<00:00,  4.43s/ episode]


  Probability of finding correct solution: 0.3200
  Average reward: -43.1534457578862
  Lowest reward obtained: -47.019716095227274
  Best state at lowest value: [-1 -1  1  1 -1  1  1  1 -1 -1  1]
  number of nodes : 24915


Epoch 79/200:  82%|████████▏ | 41/50 [01:55<00:25,  2.81s/ episode]


KeyboardInterrupt: 